In [ ]:


import re
import json



# ----------------------------
# CONFIG
# ----------------------------
RBC_CARDS = [
    {
        "card_id": "rbc_avion_visa_infinite_ca",
        "url": "https://www.rbcroyalbank.com/credit-cards/travel/rbc-avion-visa-infinite.html"
    }
]

DATA_DIR = Path("./pythonProject/leetcode/Ai-ML-Projects/credit-card-rag/data")
DATA_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_rbc.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_rbc.jsonl"


# ----------------------------
# BROWSER
# ----------------------------
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1600,2400")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


# ----------------------------
# HELPERS
# ----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def extract_section(text: str, start_keywords, end_keywords=None, max_chars=1500):
    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])


def normalize_money(value: str) -> str:
    if not value:
        return ""
    value = value.replace("$", "").replace(",", "").strip()
    return value


def infer_best_for(page_text: str):
    text = page_text.lower()
    tags = []

    if any(x in text for x in ["travel", "flight", "airline", "hotel", "car rentals"]):
        tags.append("travel")

    if "avion" in text:
        tags.append("flexible_travel_rewards")

    if "petro-canada" in text or "fuel" in text or "gas" in text:
        tags.append("gas")

    if "doordash" in text:
        tags.append("food_delivery")

    if "hertz" in text or "car rentals" in text:
        tags.append("car_rental")

    if "visa infinite" in text:
        tags.append("premium_travel_perks")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)


def infer_one_liner(page_text: str):
    text = page_text.lower()

    if "travel" in text and "avion points" in text:
        return "Canadian travel rewards card for flexible flight redemptions, travel purchases, and premium Visa Infinite perks."

    return "Canadian rewards credit card."


def fetch_page_text(driver, url: str, wait_sec: int = 8):
    driver.get(url)
    time.sleep(wait_sec)

    # grab rendered HTML
    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))

    return html, soup, page_text


# ----------------------------
# RBC-SPECIFIC EXTRACTION
# ----------------------------
def extract_rbc_card_name(page_text: str):
    return (
        search_first(
            [
                r"(RBC Avion Visa Infinite)",
                r"(RBC Avion Visa Infinite Credit Card)"
            ],
            page_text
        )
        or "RBC Avion Visa Infinite"
    )


def extract_rbc_annual_fee(page_text: str):
    fee = search_first(
        [
            r"Annual Fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual Fee\s+\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_rbc_additional_card_fee(page_text: str):
    fee = search_first(
        [
            r"Additional Card[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Additional Card\s+\$([0-9]+(?:\.[0-9]{1,2})?)"
        ],
        page_text
    )
    return normalize_money(fee)


def extract_rbc_purchase_rate(page_text: str):
    rate = search_first(
        [
            r"Purchase Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Purchase Rate\s+([0-9]+(?:\.[0-9]{1,2})?%)"
        ],
        page_text
    )
    return rate


def extract_rbc_cash_advance_rate(page_text: str):
    rate = search_first(
        [
            r"Cash Advance Rate[:\s]*([0-9]+(?:\.[0-9]{1,2})?%[\*]?)",
            r"Cash Advance Rate\s+([0-9]+(?:\.[0-9]{1,2})?%[\*]?)"
        ],
        page_text
    )
    return rate


def extract_rbc_income_requirement(page_text: str):
    return search_first(
        [
            r"(A minimum personal income of \$[0-9,]+ or a minimum household income of \$[0-9,]+ is required for this card\.)"
        ],
        page_text
    )


def extract_rbc_rewards_summary(page_text: str):
    earn_travel = search_first(
        [r"(Earn 1\.25 Avion points on travel related purchases.*?)1X"],
        page_text
    )
    earn_other = search_first(
        [r"(Earn 1 Avion point on all other eligible purchases.*?)How to Redeem"],
        page_text
    )

    travel_rate = search_first(
        [r"(Earn 1\.25 Avion points on travel related purchases[^\.;]*)"],
        page_text
    )
    base_rate = search_first(
        [r"(Earn 1 Avion point on all other eligible purchases[^\.;]*)"],
        page_text
    )

    parts = [p for p in [travel_rate, base_rate] if p]
    return clean_text(" | ".join(parts))


def extract_rbc_welcome_bonus(page_text: str):
    # very important: RBC pages sometimes duplicate / partially hydrate this area,
    # so we keep the first strong match.
    bonus = search_first(
        [
            r"(Get up to [0-9,]+ Avion points.*?round trip flights)",
            r"(Get up to [0-9,]+ Avion points.*?)Annual Fee",
            r"(Welcome Avion Points [0-9,]+ points)"
        ],
        page_text
    )
    return clean_text(bonus)


def scrape_rbc_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_rbc_card_name(page_text)
    issuer = "RBC"
    country = "Canada"
    network = "Visa"
    card_type = "Credit Card"

    annual_fee = extract_rbc_annual_fee(page_text)
    additional_card_fee = extract_rbc_additional_card_fee(page_text)
    purchase_rate = extract_rbc_purchase_rate(page_text)
    cash_advance_rate = extract_rbc_cash_advance_rate(page_text)

    welcome_bonus_summary = extract_rbc_welcome_bonus(page_text)
    eligibility_summary = extract_rbc_income_requirement(page_text)

    best_for = infer_best_for(page_text)
    one_liner = infer_one_liner(page_text)

    overview_text = extract_section(
        page_text,
        start_keywords=[
            "RBC Avion Visa Infinite",
            "Experience Unparalleled Travel Flexibility"
        ],
        end_keywords=["Card Highlights", "How to Earn Avion Points"],
        max_chars=900
    )

    highlights_text = extract_section(
        page_text,
        start_keywords=["Card Highlights", "Book Any Flight"],
        end_keywords=["How to Earn Avion Points", "How to Redeem Avion Points"],
        max_chars=1400
    )

    rewards_text = extract_section(
        page_text,
        start_keywords=["How to Earn Avion Points"],
        end_keywords=["How to Redeem Avion Points", "See Where Your Spending Can Send You"],
        max_chars=1200
    )

    redemption_text = extract_section(
        page_text,
        start_keywords=["How to Redeem Avion Points", "Book Flights for Maximum Value"],
        end_keywords=["See Where Your Spending Can Send You", "Plan Your Next Adventure"],
        max_chars=1200
    )

    partner_offers_text = extract_section(
        page_text,
        start_keywords=["Exclusive Offers From Brands You Love"],
        end_keywords=["Discover the Infinite Possibilities with Visa Infinite", "Travel Canada in Style with Avion"],
        max_chars=1400
    )

    visa_infinite_text = extract_section(
        page_text,
        start_keywords=["Discover the Infinite Possibilities with Visa Infinite"],
        end_keywords=["Interested in exploring a credit card", "Travel Canada in Style with Avion"],
        max_chars=1400
    )

    insurance_text = extract_section(
        page_text,
        start_keywords=["Extensive Insurance Coverage", "Get Extensive Insurance"],
        end_keywords=["Even More Ways to Redeem", "Optional Add-On Services"],
        max_chars=1400
    )

    fees_text = clean_text(
        f"Annual fee: ${annual_fee}; Additional card fee: ${additional_card_fee}; "
        f"Purchase rate: {purchase_rate}; Cash advance rate: {cash_advance_rate}"
    )

    rewards_summary = extract_rbc_rewards_summary(page_text)

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": "",
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,

        # extra useful fields
        "additional_card_fee": additional_card_fee,
        "purchase_rate": purchase_rate,
        "cash_advance_rate": cash_advance_rate,
        "rewards_summary": rewards_summary
    }

    snippets = [
        {
            "chunk_id": f"{card_id}_overview",
            "card_id": card_id,
            "section": "overview",
            "text": overview_text
        },
        {
            "chunk_id": f"{card_id}_highlights",
            "card_id": card_id,
            "section": "highlights",
            "text": highlights_text
        },
        {
            "chunk_id": f"{card_id}_rewards",
            "card_id": card_id,
            "section": "rewards",
            "text": rewards_text
        },
        {
            "chunk_id": f"{card_id}_welcome_bonus",
            "card_id": card_id,
            "section": "welcome_bonus",
            "text": welcome_bonus_summary
        },
        {
            "chunk_id": f"{card_id}_redemption",
            "card_id": card_id,
            "section": "redemption",
            "text": redemption_text
        },
        {
            "chunk_id": f"{card_id}_partner_offers",
            "card_id": card_id,
            "section": "partner_offers",
            "text": partner_offers_text
        },
        {
            "chunk_id": f"{card_id}_visa_infinite",
            "card_id": card_id,
            "section": "visa_infinite_benefits",
            "text": visa_infinite_text
        },
        {
            "chunk_id": f"{card_id}_insurance",
            "card_id": card_id,
            "section": "insurance",
            "text": insurance_text
        },
        {
            "chunk_id": f"{card_id}_fees",
            "card_id": card_id,
            "section": "fees",
            "text": fees_text
        },
        {
            "chunk_id": f"{card_id}_eligibility",
            "card_id": card_id,
            "section": "eligibility",
            "text": eligibility_summary
        }
    ]

    return card_record, snippets


# ----------------------------
# RUN
# ----------------------------
driver = get_driver(headless=True)

all_cards = []
all_snippets = []

try:
    for card in RBC_CARDS:
        print(f"Scraping: {card['card_id']}")
        card_record, snippets = scrape_rbc_card(driver, card)
        all_cards.append(card_record)
        all_snippets.extend(snippets)
        time.sleep(2)
finally:
    driver.quit()


# ----------------------------
# SAVE
# ----------------------------
cards_df = pd.DataFrame(all_cards)
cards_df.to_csv(CSV_PATH, index=False)

with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
    for item in all_snippets:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved CSV to:", CSV_PATH)
print("Saved snippets to:", SNIPPETS_PATH)
print(cards_df.T)

print("\nSnippet preview:")
for s in all_snippets:
    print(f"- {s['section']}: {s['text'][:180]}...")